In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Transpilerプラグインの作成

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';





<details>
<summary><b>パッケージのバージョン</b></summary>

このページのコードは、以下の要件を使用して開発されました。
これらのバージョン以上を使用することをお勧めします。

```
qiskit[all]~=2.3.0
```
</details>

[Transpilerプラグイン](transpiler-plugins)を作成することは、Transpilationコードをより広いQiskitコミュニティと共有し、他のユーザーが開発した機能を活用できるようにする素晴らしい方法です。Qiskitコミュニティへの貢献にご関心をお寄せいただきありがとうございます！

Transpilerプラグインを作成する前に、あなたの状況に適したプラグインの種類を決める必要があります。Transpilerプラグインには3種類あります：
- [**Transpilerステージプラグイン**](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins)。プリセットのステージ付きパスマネージャーの[6つのステージ](transpiler-stages)のいずれかに代替できるパスマネージャーを定義する場合に選択してください。
- [**ユニタリー合成プラグイン**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin)。TranspilationコードがユニタリーMatrix（Numpy配列として表現）を入力として受け取り、そのユニタリーを実装する量子Circuitの記述を出力する場合に選択してください。
- [**高レベル合成プラグイン**](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin)。TranspilationコードがClifford演算子や線形関数などの「高レベルオブジェクト」を入力として受け取り、その高レベルオブジェクトを実装する量子Circuitの記述を出力する場合に選択してください。高レベルオブジェクトは[Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation)クラスのサブクラスで表されます。

作成するプラグインの種類が決まったら、以下の手順に従ってプラグインを作成してください：

1. 適切な抽象プラグインクラスのサブクラスを作成します：
   - Transpilerステージプラグインの場合は[PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin)、
   - ユニタリー合成プラグインの場合は[UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin)、
   - 高レベル合成プラグインの場合は[HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin)。
2. Pythonパッケージのメタデータ（通常は`pyproject.toml`、`setup.cfg`、または`setup.py`ファイルを編集）で、クラスを[setuptoolsエントリーポイント](https://setuptools.pypa.io/en/latest/userguide/entry_point.html)として公開します。

1つのパッケージが定義できるプラグインの数に制限はありませんが、各プラグインは固有の名前を持つ必要があります。Qiskit SDK自体にもいくつかのプラグインが含まれており、それらの名前は予約されています。予約済みの名前は以下の通りです：

- Transpilerステージプラグイン：[この表](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#plugin-stages)を参照してください。
- ユニタリー合成プラグイン：`default`、`aqc`、`sk`
- 高レベル合成プラグイン：

| 演算クラス | 演算名 | 予約済みの名前 |
| --- | --- | --- |
| [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford#clifford) | `clifford` | `default`, `ag`, `bm`, `greedy`, `layers`, `lnn` |
| [LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction#linearfunction) | `linear_function` | `default`, `kms`, `pmh` |
| [PermutationGate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.PermutationGate#permutationgate) | `permutation` | `default`, `kms`, `basic`, `acg`, `token_swapper` |

以降のセクションでは、さまざまな種類のプラグインについてこれらの手順の例を示します。これらの例では、`my_qiskit_plugin`というPythonパッケージを作成していると仮定しています。Pythonパッケージの作成については、Pythonウェブサイトの[このチュートリアル](https://packaging.python.org/en/latest/tutorials/packaging-projects/)を参照してください。

## 例：Transpilerステージプラグインの作成
この例では、`layout`ステージ用のTranspilerステージプラグインを作成します（Qiskitの組み込みTranspilationパイプラインの6つのステージの説明については[Transpilerステージ](transpiler-stages)を参照してください）。
このプラグインは、要求された最適化レベルに応じたトライアル数で[VF2Layout](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.VF2Layout)を実行するだけです。

まず、[PassManagerStagePlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin)のサブクラスを作成します。実装する必要があるメソッドは[`pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePlugin#pass_manager)と呼ばれる1つのメソッドです。このメソッドは[PassManagerConfig](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManagerConfig)を入力として受け取り、定義するパスマネージャーを返します。PassManagerConfigオブジェクトには、ターゲットBackendに関する情報（カップリングマップや基底Gateなど）が格納されています。

In [1]:
# This import is needed for python versions prior to 3.10
from __future__ import annotations

from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import VF2Layout
from qiskit.transpiler.passmanager_config import PassManagerConfig
from qiskit.transpiler.preset_passmanagers import common
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePlugin,
)


class MyLayoutPlugin(PassManagerStagePlugin):
    def pass_manager(
        self,
        pass_manager_config: PassManagerConfig,
        optimization_level: int | None = None,
    ) -> PassManager:
        layout_pm = PassManager(
            [
                VF2Layout(
                    coupling_map=pass_manager_config.coupling_map,
                    properties=pass_manager_config.backend_properties,
                    max_trials=optimization_level * 10 + 1,
                    target=pass_manager_config.target,
                )
            ]
        )
        layout_pm += common.generate_embed_passmanager(
            pass_manager_config.coupling_map
        )
        return layout_pm

次に、PythonパッケージのメタデータにエントリーポイントとしてプラグインをJで公開します。
ここでは、定義したクラスが`my_qiskit_plugin`モジュールで公開されていると仮定します（例えば、`my_qiskit_plugin`モジュールの`__init__.py`ファイルでインポートされている場合など）。
パッケージの`pyproject.toml`、`setup.cfg`、または`setup.py`ファイルを編集します（Pythonプロジェクトのメタデータを格納するために選択したファイルの種類に応じて）：

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.transpiler.layout"]
    "my_layout" = "my_qiskit_plugin:MyLayoutPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.transpiler.layout =
        my_layout = my_qiskit_plugin:MyLayoutPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.transpiler.layout': [
                'my_layout = my_qiskit_plugin:MyLayoutPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

各Transpilerステージのエントリーポイントと期待される動作については、[Transpilerプラグインステージの表](https://docs.quantum.ibm.com/api/qiskit/transpiler_plugins#stage-table)を参照してください。

プラグインがQiskitによって正常に検出されているかどうかを確認するには、プラグインパッケージをインストールし、[Transpilerプラグイン](transpiler-plugins#list-available-transpiler-stage-plugins)のインストール済みプラグインの一覧表示手順に従って、プラグインが一覧に表示されていることを確認してください：

In [2]:
from qiskit.transpiler.preset_passmanagers.plugin import list_stage_plugins

list_stage_plugins("layout")

['default', 'dense', 'sabre', 'trivial']

If our example plugin were installed, then the name `my_layout` would appear in this list.

If you want to use a built-in transpiler stage as the starting point for your transpiler stage plugin, you can obtain the pass manager for a built-in transpiler stage using [PassManagerStagePluginManager](/docs/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager). The following code cell shows how to do this to obtain the built-in optimization stage for optimization level 3.

In [3]:
from qiskit.transpiler.preset_passmanagers.plugin import (
    PassManagerStagePluginManager,
)

# Initialize the plugin manager
plugin_manager = PassManagerStagePluginManager()

# Here we create a pass manager config to use as an example.
# Instead, you should use the pass manager config that you already received as input
# to the pass_manager method of your PassManagerStagePlugin.
pass_manager_config = PassManagerConfig()

# Obtain the desired built-in transpiler stage
optimization = plugin_manager.get_passmanager_stage(
    "optimization", "default", pass_manager_config, optimization_level=3
)

サンプルプラグインがインストールされている場合、`my_layout`という名前がこの一覧に表示されます。

Transpilerステージプラグインの出発点として組み込みのTranspilerステージを使用したい場合は、[PassManagerStagePluginManager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.preset_passmanagers.plugin.PassManagerStagePluginManager#passmanagerstagepluginmanager)を使用して組み込みTranspilerステージのパスマネージャーを取得できます。以下のコードセルでは、最適化レベル3の組み込み最適化ステージを取得する方法を示します。

In [4]:
import numpy as np
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.converters import circuit_to_dag
from qiskit.dagcircuit.dagcircuit import DAGCircuit
from qiskit.quantum_info import Operator
from qiskit.transpiler.passes import UnitarySynthesis
from qiskit.transpiler.passes.synthesis.plugin import UnitarySynthesisPlugin


class MyUnitarySynthesisPlugin(UnitarySynthesisPlugin):
    @property
    def supports_basis_gates(self):
        # Returns True if the plugin can target a list of basis gates
        return True

    @property
    def supports_coupling_map(self):
        # Returns True if the plugin can synthesize for a given coupling map
        return False

    @property
    def supports_natural_direction(self):
        # Returns True if the plugin supports a toggle for considering
        # directionality of 2-qubit gates
        return False

    @property
    def supports_pulse_optimize(self):
        # Returns True if the plugin can optimize pulses during synthesis
        return False

    @property
    def supports_gate_lengths(self):
        # Returns True if the plugin can accept information about gate lengths
        return False

    @property
    def supports_gate_errors(self):
        # Returns True if the plugin can accept information about gate errors
        return False

    @property
    def supports_gate_lengths_by_qubit(self):
        # Returns True if the plugin can accept information about gate lengths
        # (The format of the input differs from supports_gate_lengths)
        return False

    @property
    def supports_gate_errors_by_qubit(self):
        # Returns True if the plugin can accept information about gate errors
        # (The format of the input differs from supports_gate_errors)
        return False

    @property
    def min_qubits(self):
        # Returns the minimum number of qubits the plugin supports
        return None

    @property
    def max_qubits(self):
        # Returns the maximum number of qubits the plugin supports
        return None

    @property
    def supported_bases(self):
        # Returns a dictionary of supported bases for synthesis
        return None

    def run(self, unitary: np.ndarray, **options) -> DAGCircuit:
        basis_gates = options["basis_gates"]
        synth_pass = UnitarySynthesis(basis_gates, min_qubits=3)
        qubits = QuantumRegister(3)
        circuit = QuantumCircuit(qubits)
        circuit.append(Operator(unitary).to_instruction(), qubits)
        dag_circuit = synth_pass.run(circuit_to_dag(circuit))
        return dag_circuit

## 例：ユニタリー合成プラグインの作成
この例では、組み込みの[UnitarySynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.UnitarySynthesis#unitarysynthesis) TranspilationパスをそのままGateの合成に使用するユニタリー合成プラグインを作成します。もちろん、実際のプラグインではこれよりも興味深い処理を行うことになります。

[UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin)クラスは、ユニタリー合成プラグインのインターフェースと規約を定義します。主要なメソッドは
[`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run)で、
ユニタリーMatrixを格納するNumpy配列を入力として受け取り、
そのユニタリーMatrixから合成されたCircuitを表す[DAGCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.dagcircuit.DAGCircuit)を返します。
`run`メソッドに加えて、定義する必要があるいくつかのプロパティメソッドがあります。
必要なすべてのプロパティのドキュメントについては[UnitarySynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin)を参照してください。

UnitarySynthesisPluginサブクラスを作成しましょう：

In [5]:
from qiskit.transpiler.passes.synthesis import unitary_synthesis_plugin_names

unitary_synthesis_plugin_names()

['aqc', 'clifford', 'default', 'gridsynth', 'sk']

[`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin#run)メソッドで利用可能な入力があなたの目的に不十分な場合は、要件を説明した[issueを開いて](https://github.com/Qiskit/qiskit/issues/new/choose)ください。追加のオプション入力の追加など、プラグインインターフェースへの変更は、既存のプラグインに変更を要求しないよう、後方互換性のある方法で行われます。

> **Note:** `supports_`で始まるすべてのメソッドは、インターフェースの一部として`UnitarySynthesisPlugin`派生クラス上で予約されています。抽象クラスで定義されていないカスタムの`supports_*`メソッドをサブクラスで定義しないでください。

次に、Pythonパッケージのメタデータにエントリーポイントとしてプラグインを公開します。
ここでは、定義したクラスが`my_qiskit_plugin`モジュールで公開されていると仮定します（例えば、`my_qiskit_plugin`モジュールの`__init__.py`ファイルでインポートされている場合など）。
パッケージの`pyproject.toml`、`setup.cfg`、または`setup.py`ファイルを編集します：

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.unitary_synthesis"]
    "my_unitary_synthesis" = "my_qiskit_plugin:MyUnitarySynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.unitary_synthesis =
        my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.unitary_synthesis': [
                'my_unitary_synthesis = my_qiskit_plugin:MyUnitarySynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

前述の通り、プロジェクトが`pyproject.toml`の代わりに`setup.cfg`または`setup.py`を使用している場合は、これらの行をあなたの状況に合わせて適応する方法について[setuptoolsのドキュメント](https://setuptools.pypa.io/en/latest/userguide/entry_point.html)を参照してください。

プラグインがQiskitによって正常に検出されているかどうかを確認するには、プラグインパッケージをインストールし、[Transpilerプラグイン](transpiler-plugins#list-available-unitary-synthesis-plugins)のインストール済みプラグインの一覧表示手順に従って、プラグインが一覧に表示されていることを確認してください：

In [6]:
from qiskit.synthesis import synth_clifford_bm
from qiskit.transpiler.passes.synthesis.plugin import HighLevelSynthesisPlugin


class MyCliffordSynthesisPlugin(HighLevelSynthesisPlugin):
    def run(
        self,
        high_level_object,
        coupling_map=None,
        target=None,
        qubits=None,
        **options,
    ) -> QuantumCircuit:
        if high_level_object.num_qubits <= 3:
            return synth_clifford_bm(high_level_object)
        else:
            return None

This plugin synthesizes objects of type [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford) that have
at most 3 qubits, using the `synth_clifford_bm` method.

Now, we expose the plugin by adding an entry point in our Python package metadata.
Here, we assume that the class we defined is exposed in a module called `my_qiskit_plugin`, for example by being imported in the `__init__.py` file of the `my_qiskit_plugin` module.
We edit the `pyproject.toml`, `setup.cfg`, or `setup.py` file of our package:

<Tabs>
  <TabItem value="package-table-toml" label="pyproject.toml" default>
    ```toml
    [project.entry-points."qiskit.synthesis"]
    "clifford.my_clifford_synthesis" = "my_qiskit_plugin:MyCliffordSynthesisPlugin"
    ```
  </TabItem>
  <TabItem value="package-table-cfg" label="setup.cfg">
    ```ini
    [options.entry_points]
    qiskit.synthesis =
        clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin
    ```
  </TabItem>
  <TabItem value="package-table-py" label="setup.py">
    ```python
    from setuptools import setup

    setup(
        # ...,
        entry_points={
            'qiskit.synthesis': [
                'clifford.my_clifford_synthesis = my_qiskit_plugin:MyCliffordSynthesisPlugin',
            ]
        }
    )
    ```
  </TabItem>
</Tabs>

The `name` consists of two parts separated by a dot (`.`):
- The name of the type of [Operation](/docs/api/qiskit/qiskit.circuit.Operation) that the plugin synthesizes (in this case, `clifford`). Note that this string corresponds to the [`name`](/docs/api/qiskit/qiskit.circuit.Operation#name) attribute of the Operation class, and not the name of the class itself.
- The name of the plugin (in this case, `special`).

As before, if your project uses `setup.cfg` or `setup.py` instead of `pyproject.toml`, see the [setuptools documentation](https://setuptools.pypa.io/en/latest/userguide/entry_point.html) for how to adapt these lines for your situation.

To check that your plugin is successfully detected by Qiskit, install your plugin package and follow the instructions at [Transpiler plugins](transpiler-plugins#list-available-high-level-synthesis-plugins) for listing installed plugins, and ensure that your plugin appears in the list:

In [7]:
from qiskit.transpiler.passes.synthesis import (
    high_level_synthesis_plugin_names,
)

high_level_synthesis_plugin_names("clifford")

['ag', 'bm', 'default', 'greedy', 'layers', 'lnn', 'rb_default']

サンプルプラグインがインストールされている場合、`my_unitary_synthesis`という名前がこの一覧に表示されます。

複数のオプションを公開するユニタリー合成プラグインに対応するため、
プラグインインターフェースにはユーザーが自由形式の
設定辞書を提供するオプションがあります。これは`options`キーワード引数を通じて`run`メソッドに渡されます。プラグインにこのような設定オプションがある場合は、それらを明確にドキュメント化する必要があります。

## 例：高レベル合成プラグインの作成

この例では、組み込みの[synth_clifford_bm](https://docs.quantum.ibm.com/api/qiskit/synthesis#synth_clifford_bm)関数をそのまま使用してClifford演算子を合成する高レベル合成プラグインを作成します。

[HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin)クラスは、高レベル合成プラグインのインターフェースと規約を定義します。主要なメソッドは[`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin#run)です。
位置引数`high_level_object`は、合成される「高レベル」オブジェクトを表す[Operation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Operation)です。例えば、
[LinearFunction](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction)や
[Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford)などが該当します。
以下のキーワード引数が存在します：
- `target`はターゲットBackendを指定し、カップリングマップ、サポートされているGateセットなど、ターゲット固有のすべての情報にプラグインがアクセスできるようにします。
- `coupling_map`はカップリングマップのみを指定し、`target`が指定されていない場合にのみ使用されます。
- `qubits`は高レベルオブジェクトが定義されているQubitのリストを指定します。物理Circuitで合成が行われる場合に使用されます。
``None``の値は、レイアウトがまだ選択されておらず、このオペレーションが動作するターゲットまたはカップリングマップの物理Qubitがまだ決定されていないことを示します。
- `options`は、プラグイン固有のオプションのための自由形式の設定辞書です。プラグインにこのような設定オプションがある場合は、それらを明確にドキュメント化する必要があります。

`run`メソッドは、その高レベルオブジェクトから合成されたCircuitを表す[QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit)を返します。
`None`を返すことも許可されており、その場合はプラグインが指定された高レベルオブジェクトを合成できないことを示します。
高レベルオブジェクトの実際の合成は、
[HighLevelSynthesis](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.HighLevelSynthesis)
Transpilerパスによって実行されます。

`run`メソッドに加えて、定義する必要があるいくつかのプロパティメソッドがあります。
必要なすべてのプロパティのドキュメントについては[HighLevelSynthesisPlugin](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin)を参照してください。

HighLevelSynthesisPluginサブクラスを定義しましょう：